在Hello- Agents项目中的AutoGen 软件开发团队协作案例进行优化扩展

@author kang

- 将Openai模型替换为本地Ollama模型，避免依赖网络
- 添加模型选择器，根据任务类型选择不同的模型,将RoundRobinGroupChat替换为SelectorGroupChat
- 使用懒加载模型池，避免重复创建模型
- 添加模型释放功能，确保模型在不需要时被释放
- 添加模型配置参数，支持自定义模型配置
- 添加shell脚本，用于启动和关闭容器，测试代码是否正常运行


In [1]:
import os
import sys
sys.path.append("./")
from utils import local_server_url, print_markdown
from testing_env.shell_debug import shell_debug
import asyncio
from typing import List, Dict, Any
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# 先测试一个版本，使用 ollama 客户端
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.teams import RoundRobinGroupChat,SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console

定义懒加载模型池

In [2]:
class LazyModelPool:
    """懒加载模型池"""
    def __init__(self):
        self.models = {}
    
    def get_model(self, model_name: str,**kwargs):
        """需要时才创建模型"""
        if model_name not in self.models:
            # 只在第一次请求时创建
            self.models[model_name] = create_ollama_model_client(model_name,**kwargs)   
        return self.models[model_name]
    
    async def close_all(self):
        """关闭所有模型客户端"""
        for model_name, model in self.models.items():
            try:
                await model.close()
                print(f"✅ 模型 {model_name} 已关闭")
            except Exception as e:
                print(f"❌ 关闭模型 {model_name} 失败: {e}")
        self.models.clear()

创建ollama模型客户端及模型配置

In [3]:
def create_ollama_model_client(model_name: str = "qwen3.5:35b",**kwargs):
    """创建 ollama 模型客户端用于测试"""
    return OllamaChatCompletionClient(
        model=model_name,
        api_key=os.getenv("LLM_API_KEY","sk-no-api-key"),
        base_url=os.getenv("LLM_BASE_URL", local_server_url),
        model_info=create_model_config(model_name,**kwargs)
    )

def create_model_config(model_name: str = "qwen3.5:35b",**kwargs):
    """创建 ollama 模型配置"""
    return {
        "model": model_name,
        "context_window":kwargs.get("context_window",4096),
        "temperature":kwargs.get("temperature",0.7),
        "top_p":kwargs.get("top_p",0.95),
        "repeat_penalty":kwargs.get("repeat_penalty",0.0),
        "presence_penalty":kwargs.get("presence_penalty",0.0),
        "is_chat_model":kwargs.get("is_chat_model",True),
        "json_output":kwargs.get("json_output",False),
        "vision":kwargs.get("vision",False),
        "function_calling":kwargs.get("function_calling",False),
        "family":kwargs.get("family","Qwen")
    }  

创建开发团队角色

In [4]:
def create_product_manager(model_client):
    """创建产品经理智能体"""
    system_message = f"""你是一位经验丰富的产品经理（ProductManager），专门负责软件产品的需求分析和项目规划。
    团队里面其他智能体包括：Engineer、CodeReviewer、CodeTester、UserProxy。
    
    你的核心职责包括：
    1. **需求分析**：深入理解用户需求，识别核心功能和边界条件
    2. **技术规划**：基于需求制定清晰的技术实现路径
    3. **风险评估**：识别潜在的技术风险和用户体验问题
    4. **协调沟通**：与工程师和其他团队成员进行有效沟通

    当接到开发任务时，请按以下结构进行分析：
    1. 需求理解与分析
    2. 功能模块划分
    3. 技术选型建议
    4. 实现优先级排序
    5. 验收标准定义
    API代码及返回结果请参考：{read_code('example.py')}
    不直接参与代码实现，仅负责需求分析和项目规划。
    请简洁明了地回应，并在分析完成后说"请工程师开始实现"。"""

    return AssistantAgent(
        name="ProductManager",
        model_client=model_client,
        system_message=system_message,
    )

def create_engineer(model_client):
    """创建软件工程师智能体"""
    system_message = f"""你是一位资深的软件工程师（Engineer），擅长 Python 开发和 Web 应用构建。
    团队里面其他智能体包括：ProductManager、CodeReviewer、CodeTester、UserProxy。

    你的技术专长包括：
    1. **Python 编程**：熟练掌握 Python 语法和最佳实践
    2. **Web 开发**：精通 Streamlit、Flask、Django 等框架
    3. **API 集成**：有丰富的第三方 API 集成经验
    4. **错误处理**：注重代码的健壮性和异常处理

    当收到开发任务时，请：
    1. 仔细分析技术需求
    2. 选择合适的技术方案
    3. 编写完整的代码实现
    4. 添加必要的注释和说明
    5. 考虑边界情况和异常处理
    6.API代码及返回结果请参考：{read_code('example.py')}
    
    完成代码构建后：
    1. 保存代码到 ./testing_env 目录下的 output.py 文件中
    2. 通知代码审查员检查代码
    """

    return AssistantAgent(
        name="Engineer",
        model_client=model_client,
        system_message=system_message,
        tools=[save_code],
    )

def create_code_reviewer(model_client):
    """创建代码审查员智能体"""
    system_message = f"""你是一位经验丰富的代码审查专家（CodeReviewer），专注于代码质量和最佳实践。
    团队里面其他智能体包括：ProductManager、Engineer、CodeTester、UserProxy。

    你的审查重点包括：
    1. **代码质量**：检查代码的可读性、可维护性和性能
    2. **安全性**：识别潜在的安全漏洞和风险点
    3. **最佳实践**：确保代码遵循行业标准和最佳实践
    4. **错误处理**：验证异常处理的完整性和合理性

    **注意**：不直接参与代码实现，仅负责审查代码质量和最佳实践。
    审查流程：
    1. 仔细阅读和理解代码逻辑
    2. 检查代码规范和最佳实践
    3. 识别潜在问题和改进点
    4. 提供具体的修改建议
    5. 评估代码的整体质量
    
    请提供具体的审查意见，
    完成后如果代码审查通过，说"代码审查完成，请CodeTester测试"。
    如果代码审查不通过，说"代码审查不通过，请Engineer修复代码"。
    """

    return AssistantAgent(
        name="CodeReviewer",
        model_client=model_client,
        system_message=system_message,
    )

def create_code_tester(model_client):
    """创建代码测试智能体"""
    system_message = f"""
    你是一位经验丰富的代码测试专家（CodeTester），专注于测试代码的功能和性能，不直接参与代码实现。
    团队里面其他智能体包括：ProductManager、Engineer、CodeReviewer、UserProxy。

    测试流程：
    1. 调用工具shell_debug进行代码测试，查看代码是否正常运行。
    2. 如果返回错误信息，将错误信息告知Engineer,并要求Engineer修复代码。
    3. 如果测试通过，将测试结果告知UserProxy。
    """
    return AssistantAgent(
        name="CodeTester",
        model_client=model_client,
        system_message=system_message,
        tools=[shell_debug],
    )

def create_user_proxy():
    """创建用户代理智能体"""
    return UserProxyAgent(
        name="UserProxy",
        description="""用户代理，负责以下职责：
        1. 代表用户提出开发需求
        2. 执行最终的代码实现
        3. 验证功能是否符合预期
        4. 提供用户反馈和建议
        5. 根据用户反馈建议，选择结束或者继续开发

        完成测试后请回复 TERMINATE。""",
    )

创建工具

In [5]:
# 定义保存代码的工具函数
def save_code(code: str, filename: str) -> str:
    """
    保存代码到文件
    code: 要保存的代码
    filename: 保存的文件名
    返回：保存结果
    例如：✅ 代码已成功保存到 ./testing_env/output.py，交由用户代理审查。
    例如：❌ 保存代码失败：[错误信息]，交由用户代理处理测试。
    """
    if not os.path.exists("./testing_env"):
       os.makedirs("./testing_env")
    try:
        file_path = os.path.join("./testing_env",filename) if filename.startswith("output") else filename
        with open(file_path, 'w', encoding='utf-8') as f:
            f.write(code)
        return f"✅ 代码已成功保存到 {file_path}，代码内容是：{code},交由CodeReviewer审查。"
    except Exception as e:
        return f"❌ 保存代码失败：{e}，交由UserProxy处理。"

def read_code(filename: str) -> str:
    """
    读取代码文件工具函数
    filename: 要读取的代码文件名
    返回：代码内容
    """
    try:
        with open(os.path.join("./testing_env",filename), 'r', encoding='utf-8') as f:
            code = f.read()
    except Exception as e:
        return f"❌ 读取代码文件失败：{e},请交由UserProxy处理"  
    return code

组织团队协作

In [6]:
async def run_software_development_team(item:str='黄金',measurement:str='USD/盎司',model_pool:LazyModelPool=None):
    """运行软件开发团队协作"""
    
    print("🔧 正在初始化模型客户端...")
    # 先使用标准的 ollama 客户端测试
    general_model_client = model_pool.get_model("qwen3.6:35b",temperature=0.7,repeat_penalty=1.1)
    engineer_model_client = model_pool.get_model("qwen3-coder:30b",function_calling=True,temperature=0.7,repeat_penalty=1.1)
    # code_reviewer_model_client = model_pool.get_model("deepseek-r1:32b",context_window=8192)
    print("👥 正在创建智能体团队...")
    
    # 创建智能体团队
    product_manager = create_product_manager(general_model_client)
    engineer = create_engineer(engineer_model_client)
    code_reviewer = create_code_reviewer(general_model_client)
    code_tester = create_code_tester(engineer_model_client)
    user_proxy = create_user_proxy()
    
    # 添加终止条件
    termination = TextMentionTermination("TERMINATE")

    # 定义选择器提示
    selector_prompt = """
    选择一个智能体来执行任务。

    {roles}

    当前对话上下文：
    {history}

    阅读上述对话，然后从 {participants} 中选择一个智能体来执行下一个任务。
    确保规划智能体在其他智能体开始工作之前已经分配了任务。
    只能选择一个智能体。

    一个完整的开发流程必须包括以下步骤：
    1. 产品经理提出需求，规划智能体分配需求。
    2. 工程师根据需求进行代码实现。
    3. 代码审查智能体审查代码，确保符合需求。
    4. 代码测试智能体测试代码，确保功能正常。
    5. 用户代理验证功能是否符合预期。
    执行步骤按照对话最后的提示进行，不一定按顺序进行。
    """
    # 创建团队聊天
    team_chat = SelectorGroupChat(
        participants=[
            product_manager,
            engineer, 
            code_reviewer,
            code_tester,
            user_proxy
        ],
        model_client=general_model_client,
        termination_condition=termination,
        selector_prompt=selector_prompt,
        max_turns=100,  # 增加最大轮次
    )
    
    # 定义开发任务
    task = f"""我们需要开发一个{item}价格显示应用，具体要求如下：

    核心功能：
    - 实时显示{item}当前价格{measurement}
    - 显示24小时价格变化趋势（涨跌幅和涨跌额）
    - 提供价格刷新功能

    技术要求：
    - 使用 Streamlit 框架创建 Web 应用
    - 界面简洁美观，用户友好
    - 添加适当的错误处理和加载状态

    请团队协作完成这个任务，从需求分析到最终实现。"""
    
    # 执行团队协作
    print("🚀 启动 AutoGen 软件开发团队协作...")
    print("=" * 60)
    
    # 使用 Console 来显示对话过程
    result = await Console(team_chat.run_stream(task=task))
    
    print("\n" + "=" * 60)
    print("✅ 团队协作完成！")

    await model_pool.close_all()
    return result

主程序

In [7]:
async def main():
    """主程序入口"""
    model_pool = LazyModelPool()
    try:
        # 运行异步协作流程
        #result = asyncio.run(run_software_development_team(item='Au(T+D)',measurement='元/克',model_pool=model_pool))
        result = await run_software_development_team(item='Au(T+D)',measurement='元/克',model_pool=model_pool)
        
        print(f"\n📋 协作结果摘要：")
        print(f"- 参与智能体数量：4个")
        print(f"- 任务完成状态：{'成功' if result else '需要进一步处理'}")
        
    except ValueError as e:
        print(f"❌ 配置错误：{e}")
        print("请检查 .env 文件中的配置是否正确")
    except Exception as e:
        print(f"❌ 运行错误：{e}")
        import traceback
        traceback.print_exc()
    finally:
        import gc
        gc.collect()

In [ ]:
await main()

🔧 正在初始化模型客户端...
👥 正在创建智能体团队...
🚀 启动 AutoGen 软件开发团队协作...
---------- TextMessage (user) ----------
我们需要开发一个Au(T+D)价格显示应用，具体要求如下：

    核心功能：
    - 实时显示Au(T+D)当前价格元/克
    - 显示24小时价格变化趋势（涨跌幅和涨跌额）
    - 提供价格刷新功能

    技术要求：
    - 使用 Streamlit 框架创建 Web 应用
    - 界面简洁美观，用户友好
    - 添加适当的错误处理和加载状态

    请团队协作完成这个任务，从需求分析到最终实现。
---------- TextMessage (ProductManager) ----------
### 1. 需求理解与分析
- **核心目标**：构建轻量级、实时可靠的Au(T+D)价格监控看板，满足用户对金价动态的即时查看需求。
- **数据源对接**：基于提供的上海黄金交易所接口，需精准提取`Au(T+D)`品种数据。注意API返回的`limit`（涨跌幅）带`%`符号，`NaN`及`--`需做安全清洗；涨跌额需通过 `(最新价 - 昨收价)` 推导。
- **边界与风险**：① 非交易时段数据可能为空（`--`或`NaN`）；② 第三方API限流或超时；③ Streamlit的重渲染机制易导致页面闪烁，需合理配置缓存与状态。
- **24小时趋势说明**：当前API仅返回实时快照，本版本将实现“今日开盘价 vs 当前价 vs 昨收价”的视觉化趋势指示器，并预留历史分时/K线接口扩展能力。

### 2. 功能模块划分
- **数据层**：API请求封装、参数配置（apiKey环境变量）、JSON解析、空值/异常清洗、TTL缓存策略。
- **业务层**：价格格式化（元/克，保留2位）、涨跌幅计算与正负号映射、趋势状态判定（涨/跌/平）、单位换算。
- **UI交互层**：核心价格卡片、涨跌幅指标、手动刷新按钮、加载动画(`st.spinner`)、异常状态提示(`st.error`/`st.warning`)。
- **控制层**：手动触发刷新、自动轮询定时器、环境变量读取、Session状态管理（防重复请求）。

### 3. 技